# CRT III - Assignment 1: Parameter Estimation
Benjamin Heuschmid , Sanskar Nashier, Maximilian Franz (Group 2)

## 1 Introduction

######## NEEDS CHANGING
In chemical engineering, parameter fittings are essential. If, for example, a polytropically operated batch reactor is simulated, it is important to have information about the heat transfer coefficients [1], the thermophysical properties of the fluids [2] or the reaction kinetics [3]. If the reaction takes place in a heterogeneous system, the mass transport between the phases also plays a decisive role [4]. These simulations can be used for safety [1], scale-up or optimization of the reactor output [5].
However, since these parameters are usually not measurable, they must be obtained by a parameter estimation. For this purpose, a model is set up in which the parameters are fitted to the experimentally obtained results. If the accuracy of the fit to the experimental data is low, the model has to be modified, and the accuracy must be verified again.

Using experimentally obtained concentration profiles of an isothermally operated batch stirred tank reactor, a reaction network and reaction kinetics are proposed and evaluated. For this purpose, concentration profiles at different initial concentrations are analyzed, and a reaction hypothesis for a reaction network is developed. The proposed reaction kinetics are assessed by comparison with the experimentally obtained data using parameter fitting of the reaction rate constants.   

#############

## 2 Experimental Data

################NEEDS CHANGING
In the first step, all needed packages are imported and the path to where the experimental data is stored is defined.

In [ ]:
#%%
# Imports of packages
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import ICIW_Plots.colors as iciw_colors
plt.style.use("ICIWstyle")
import scipy.integrate as integ
from lmfit import Model, Minimizer, Parameters, report_fit

# Path to folder with csv-data
path = "2026_Task_data"


get exp data from csv files and make it usable

In [ ]:
# create list of files in directory
os_list = os.listdir(path)
# provide length of list, which equals the number of experiments (nex)
nex = len(os_list)

# create empty list for concentrations, times and initials to store experimental results
exp_concs = []
t_eval = []
c_inits = []
for i in np.arange(0, nex):
    # iterate over all csv-files and append them, after transposing, as array to the list
    aux = pd.read_csv(path + r'\Exp_res_' + str(i) + '.csv')
    t_eval.append(aux.values.transpose()[0, :])         # time in s
    c_inits.append(aux.values.transpose()[1:, 0])       # initial concentration of component i in mol/m^3
    exp_concs.append(aux.values.transpose()[1:, :])     # timedependent concentration of component i in mol/m^3

# Create flattened array of results
nex = len(exp_concs)
exp_concs_flat = np.array([])
for i in np.arange(0, nex):
    exp_concs_flat = np.append(exp_concs_flat, exp_concs[i])


plot exp data, added cumulative concentration to see stochiometry

In [ ]:
#%%
# Automatic plotting of all experiments

components = ["A", "B", "C", "D", "E", "F"]

colors = {
    "A": iciw_colors.CRIMSON,
    "B": iciw_colors.CERULEAN,
    "C": iciw_colors.KELLYGREEN,
    "D": iciw_colors.FLAME,
    "E": iciw_colors.DRAB,
    "F": "magenta"
}

for exp in range(1, nex + 1):

    exp_idx = exp - 1

    fig, ax = plt.subplots(figsize=(10,10))
    cumulative = 0

    for comp in components:

        comp_idx = components.index(comp)

        ax.scatter(
            t_eval[exp_idx],
            exp_concs[exp_idx][comp_idx, :],
            label=f"Concentration {comp} Exp. {exp}",
            color=colors[comp],
            marker="o"
        )

        cumulative += exp_concs[exp_idx][comp_idx, :]

    # Plot cumulative concentration
    ax.scatter(
        t_eval[exp_idx],
        cumulative,
        label=f"Cumulative Concentration Exp. {exp}",
        color="cyan",
        marker="o"
    )

    ax.set_xlabel("t / s")
    ax.set_ylabel(r'$\mathrm{c}\; / \ \mathrm{\frac{mol}{m^3}}$')
    ax.legend()
    ax.set_title(f"Experiment {exp}")

    plt.show()


## 3 Hypothesis
#### 3.1 guessing from exp data
### Exp 1